# Multi-Tool Order Support Agent

This notebook is a hands-on exercise for practicing LangChain tool calling with multiple tools.

The goal is to implement the full tool-calling loop yourself:

1. Define tools with `@tool`.
2. Bind tools to an LLM with `bind_tools`.
3. Ask a user question that may require one or more tools.
4. Inspect `tool_request.tool_calls`.
5. Execute the requested tools dynamically.
6. Return tool results to the LLM with `ToolMessage`.
7. Ask the LLM for a final user-facing answer.


## Exercise Objective

Build a simple order-support assistant that can use multiple tools.

The assistant should be able to:

- Check an order status.
- Calculate a refund amount.
- Create a support ticket.
- Estimate a delivery delay.
- Generate a confirmation code.

Do not hard-code which tool to run for each question. Let the LLM request tools, then use your Python dispatcher to execute the requested tools.

## Step 1: Imports And Model Setup

Import the packages you need and create your base LLM.

Requirements:

- Load `OPENAI_API_KEY` from `config.json`.
- Import `ChatOpenAI`.
- Import `tool`.
- Import `HumanMessage` and `ToolMessage`.
- Create an LLM with `temperature=0`.

In [9]:
# TODO: Add imports and model setup here.

import sqlite3
import json
import os
from pathlib import Path

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

config_path = Path("config.json")
with config_path.open(encoding="utf-8") as config_file:
    config = json.load(config_file)

os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

DB_PATH = "kartify.db"

## Step 2: Define The Tools

Implement the five tools below with `@tool`.

Each tool should have:

- A clear function name.
- Typed parameters where possible.
- A useful docstring. The LLM uses this docstring to decide when to call the tool.
- A simple return value.

### Tool 1: `get_order_status`

Responsibility: return order information for a given `order_id`.

Input:

- `order_id`: string

Output:

- A string with order status, product, shipping partner, tracking number, and estimated delivery.

Implementation options:

- Use a small in-memory dictionary, or
- Query `kartify.db`.

In [10]:
DB_PATH = "kartify.db"

@tool
def get_order_status(order_id: str) -> str:
    """Get order status details from the Kartify SQLite database for a given order ID, such as O40327."""
    import sqlite3

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT
            order_id,
            customer_id,
            product_description,
            payment_status,
            order_status,
            tracking_number,
            shipping_partner,
            dispatch_date,
            expected_delivery,
            actual_delivery
        FROM orders
        WHERE order_id = ?
        """,
        (order_id,)
    )

    row = cursor.fetchone()
    conn.close()

    if row is None:
        return f"No order found for order ID {order_id}."

    (
        order_id,
        customer_id,
        product_description,
        payment_status,
        order_status,
        tracking_number,
        shipping_partner,
        dispatch_date,
        expected_delivery,
        actual_delivery,
    ) = row

    return (
        f"Order ID: {order_id}. "
        f"Customer ID: {customer_id}. "
        f"Product: {product_description}. "
        f"Payment status: {payment_status}. "
        f"Order status: {order_status}. "
        f"Tracking number: {tracking_number}. "
        f"Shipping partner: {shipping_partner}. "
        f"Dispatch date: {dispatch_date}. "
        f"Expected delivery: {expected_delivery}. "
        f"Actual delivery: {actual_delivery or 'Not delivered yet'}."
    )


### Tool 2: `calculate_refund`

Responsibility: calculate a refund after subtracting a processing fee.

Inputs:

- `price`: number
- `fee_percent`: number

Output:

- Final refund amount.

In [ ]:
@tool
def calculate_refund(price: float, fee_percent: float) -> float:
    """Calculate the final refund amount after subtracting a processing fee percentage."""
    fee_amount = price * (fee_percent / 100)
    final_refund = price - fee_amount
    return round(final_refund, 2)


### Tool 3: `create_support_ticket`

Responsibility: simulate creating a support ticket.

Input:

- `reason`: string

Output:

- A string with a fake ticket ID and the reason.

In [ ]:
# TODO: Implement create_support_ticket with @tool.
def create_support_ticket(order_id: str, issue_description: str) -> str:
    """Create a support ticket for a given order ID and issue description."""
    # Placeholder implementation. Replace with actual ticket creation logic.
    return f"Support ticket created for order {order_id} with issue: {issue_description}."



### Tool 4: `estimate_delivery_date`

Responsibility: estimate a revised delivery day or date after a delay.

Inputs:

- `current_delivery_day`: number or date string
- `extra_days`: number

Output:

- New estimated delivery day or date.

In [ ]:
@tool
def estimate_delivery_date(current_delivery_day: int | str, extra_days: int) -> int | str:
    """Estimate a new delivery day or date by adding extra delay days."""
    from datetime import datetime, timedelta

    if isinstance(current_delivery_day, int):
        return current_delivery_day + extra_days

    current_delivery_day = str(current_delivery_day).strip()

    for date_format in ("%Y-%m-%d", "%d-%m", "%m-%d"):
        try:
            parsed_date = datetime.strptime(current_delivery_day, date_format)
            new_date = parsed_date + timedelta(days=extra_days)
            return new_date.strftime(date_format)
        except ValueError:
            pass

    return f"Could not parse delivery date: {current_delivery_day}."


### Tool 5: `generate_confirmation_code`

Responsibility: generate a short confirmation code.

Input:

- Optional `prefix`: string

Output:

- A confirmation code such as `SUP-48392`.

In [ ]:
# TODO: Implement generate_confirmation_code with @tool.
@tool
def generate_confirmation_code(order_id: str) -> str:
    """Generate a unique confirmation code for a given order ID."""
    import hashlib

    hash_object = hashlib.sha256(order_id.encode())
    confirmation_code = hash_object.hexdigest()[:10]
    return confirmation_code



## Step 3: Bind Tools To The LLM

Create a list of all tools and bind them to the LLM.

Checkpoint:

- You should have a base LLM.
- You should have a new LLM instance with tools bound to it.
- You should understand that `bind_tools` makes the tools available to the model, but it does not execute them.

In [ ]:
# TODO: Create a tools list and bind it to the LLM.



## Step 4: Ask A Multi-Tool Question

Write a user question that should require multiple tools.

Example intent:

- Check an order.
- Calculate a refund.
- Create a ticket.
- Generate a confirmation code.

Then invoke the LLM with tools and inspect `tool_request.tool_calls`.

Important: this step identifies requested tools. It does not execute them.

In [ ]:
# TODO: Create a user question and call llm_with_tools.invoke(...).
# TODO: Print or inspect tool_request.tool_calls.



## Step 5: Execute Requested Tools Dynamically

Build a dispatcher that executes whatever tools the LLM requested.

Requirements:

- Do not use a long `if/elif` chain.
- Create a dictionary that maps tool names to tool objects.
- Loop over `tool_request.tool_calls`.
- For each tool call, find the correct tool by name.
- Execute it with `.invoke(call["args"])`.
- Wrap the result in a `ToolMessage`.

Checkpoint:

- You should be able to print each tool name, arguments, and result.

In [ ]:
# TODO: Build the dynamic tool dispatcher.
# TODO: Execute requested tools and create ToolMessage objects.



## Step 6: Generate The Final Answer

Send the conversation back to the LLM with:

- The original user message.
- The LLM message that requested tool calls.
- The `ToolMessage` results.

The final response should be a natural-language answer that uses the tool outputs.

In [ ]:
# TODO: Ask the LLM for the final answer using the tool results.



## Step 7: Test Cases

Run at least these scenarios:

1. One tool: ask only for order status.
2. Two tools: ask for order status and refund calculation.
3. Three or more tools: ask for order status, refund, ticket, and confirmation code.
4. No tool needed: say `Thanks, that is all.`
5. Ambiguous request: say `I need help with my order.`

For each test, inspect whether the LLM requested the expected tools.

In [ ]:
# TODO: Add your test cases here.



## Reflection Questions

Answer these after you finish:

1. What does `@tool` do?
2. What does `bind_tools` do?
3. What is inside `tool_request.tool_calls`?
4. Why does `llm_with_tools.invoke()` not execute the Python functions directly?
5. How does your code decide which function to execute?
6. Why does the final LLM call need `ToolMessage` objects?
7. What happens when the LLM requests no tools?
8. What happens when the LLM requests multiple tools?